In [3]:
import os
os.chdir("../")

In [7]:
from tokenizer import BPETokenizer
tokenizer = BPETokenizer()
tokenizer.load("data/vocab.json", "data/merges.json")

In [8]:
with open(r"D:\GptFromScratch\data\the_verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [9]:
encoded_text = tokenizer.encode(raw_text)

In [10]:
len(encoded_text)

6068

In [11]:
encoded_sample = encoded_text[50:]

In [14]:
import torch
from torch.utils.data import Dataset, DataLoader

In [38]:
class TextDataset(Dataset):
    def __init__(self, tokenizer, txt, max_length, stride):
        self.tokenizer = tokenizer
        self.txt = txt
        self.max_length = max_length
        self.stride = stride
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(txt) # Encode entire text into token IDs
        
        for i in range(0, len(token_ids) - max_length, stride):
            input_seq = token_ids[i:i + max_length]
            target_seq = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_seq))
            self.target_ids.append(torch.tensor(target_seq))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [39]:
def create_dataloader(txt, batch_size, max_length, stride, shuffle=True,num_workers=0, drop_last=True):
    
    tokenizer = BPETokenizer()
    tokenizer.load("data/vocab.json", "data/merges.json")
    dataset = TextDataset(tokenizer, txt, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, drop_last=drop_last)
    return dataloader

In [40]:
dataloader = create_dataloader(raw_text, batch_size=16, max_length=512, stride=256)

In [41]:
data_iter = iter(dataloader)
next_batch = next(data_iter)
print(next_batch[0].shape)  # Input batch shape
print(next_batch[1].shape)  # Target batch shape

torch.Size([16, 512])
torch.Size([16, 512])
